In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    #.set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.0.0")
    #.set("spark.executor.heartbeatInterval", "300000")
    #.set("spark.network.timeout", "400000")
    #.set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    #.set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    #.set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    #.set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    #.set("spark.sql.warehouse.dir", "s3a://defaultfs/spark/warehouse")
    #.set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

spark = (
    SparkSession.builder.master("local[*]").
        appName('spark-null-handling').
        config(conf=sparkConf).
        getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/06 07:10:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
payment_col= ["paymentId", "customerId","amount"]

payment_data = [
    (1, 101, 2500), 
    (2, 102, None), 
    (3, 103, 500), 
    (4, 104, 400), 
    (5, None, 150), 
    (6, 106, None)
]

payment_df = spark.createDataFrame(payment_data,payment_col)
payment_df.show()

customer_col = ["customerId", "name"]

customer_data = [
    (101,"Jon"), 
    (102,"Aron"),
    (103,"Sam"),
    (106,None),
    (None,"Manik"),
    (108,"Leo")
]

customer_df = spark.createDataFrame(customer_data,customer_col)
customer_df.show()

+---------+----------+------+
|paymentId|customerId|amount|
+---------+----------+------+
|        1|       101|  2500|
|        2|       102|  NULL|
|        3|       103|   500|
|        4|       104|   400|
|        5|      NULL|   150|
|        6|       106|  NULL|
+---------+----------+------+

+----------+-----+
|customerId| name|
+----------+-----+
|       101|  Jon|
|       102| Aron|
|       103|  Sam|
|       106| NULL|
|      NULL|Manik|
|       108|  Leo|
+----------+-----+



In [10]:
leftJoinDf = payment_df.join(customer_df, customer_df["customerId"] == payment_df["customerId"], "left" )
leftJoinDf.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  NULL|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|      NULL|   150|      NULL|NULL|
|        6|       106|  NULL|       106|NULL|
+---------+----------+------+----------+----+



In [ ]:
#
# 'customerId' is the common column name in both payment_df and customer_df
#

joined_df = payment_df.join(customer_df, ["customerId"], "left" )
joined_df.show()

# Resulting DataFrame will have only one 'customerId' column

+----------+---------+------+----+
|customerId|paymentId|amount|name|
+----------+---------+------+----+
|       101|        1|  2500| Jon|
|       102|        2|  NULL|Aron|
|       103|        3|   500| Sam|
|       104|        4|   400|NULL|
|      NULL|        5|   150|NULL|
|       106|        6|  NULL|NULL|
+----------+---------+------+----+



In [45]:
#
# drop duplicate column
#
resultDf = leftJoinDf.drop(customer_df.customerId)
resultDf.show()

+---------+----------+------+----+
|paymentId|customerId|amount|name|
+---------+----------+------+----+
|        1|       101|  2500| Jon|
|        2|       102|  NULL|Aron|
|        3|       103|   500| Sam|
|        4|       104|   400|NULL|
|        5|      NULL|   150|NULL|
|        6|       106|  NULL|NULL|
+---------+----------+------+----+



In [55]:
#
# Rename column in customer_df 
#
resultDf = leftJoinDf.select(\
    payment_df['paymentId'], \
        payment_df['customerId'], \
        customer_df['name'], \
        customer_df['customerId'].alias('cust_id') 
    )
resultDf.show()

+---------+----------+----+-------+
|paymentId|customerId|name|cust_id|
+---------+----------+----+-------+
|        1|       101| Jon|    101|
|        2|       102|Aron|    102|
|        3|       103| Sam|    103|
|        4|       104|NULL|   NULL|
|        5|      NULL|NULL|   NULL|
|        6|       106|NULL|    106|
+---------+----------+----+-------+



In [ ]:
#
# Select the column at index 1
#

index_to_select = 2
cols = resultDf.columns
print(cols[index_to_select])
resultDf.select(resultDf[cols[index_to_select]]).show()


name
+----+
|name|
+----+
| Jon|
|Aron|
| Sam|
|NULL|
|NULL|
|NULL|
+----+



In [ ]:
#
# Exclude the column at index 1
#

index_to_exclude = 1
cols_to_keep = resultDf.columns[:index_to_exclude] + resultDf.columns[index_to_exclude+1:]
resultDf.select(cols_to_keep).show()


+---------+----+-------+
|paymentId|name|cust_id|
+---------+----+-------+
|        1| Jon|    101|
|        2|Aron|    102|
|        3| Sam|    103|
|        4|NULL|   NULL|
|        5|NULL|   NULL|
|        6|NULL|    106|
+---------+----+-------+



In [58]:
# Define the indices you want
indices_to_select = [0, 2, 3]

# Get the corresponding column names
selected_cols = [resultDf.columns[i] for i in indices_to_select]

# Select these columns
resultDf.select(*selected_cols).show()


+---------+----+-------+
|paymentId|name|cust_id|
+---------+----+-------+
|        1| Jon|    101|
|        2|Aron|    102|
|        3| Sam|    103|
|        4|NULL|   NULL|
|        5|NULL|   NULL|
|        6|NULL|    106|
+---------+----+-------+



In [ ]:
#
# Select the first three columns (indices 0 to 2)
#
resultDf.select(resultDf.columns[:3]).show()

+---------+----------+----+
|paymentId|customerId|name|
+---------+----------+----+
|        1|       101| Jon|
|        2|       102|Aron|
|        3|       103| Sam|
|        4|       104|NULL|
|        5|      NULL|NULL|
|        6|       106|NULL|
+---------+----------+----+



In [11]:
from pyspark.sql.functions import asc_nulls_last

# Sort "department" ascending, with nulls at the end
leftJoinDf.sort(asc_nulls_last("name")).show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        2|       102|  NULL|       102|Aron|
|        1|       101|  2500|       101| Jon|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|      NULL|   150|      NULL|NULL|
|        6|       106|  NULL|       106|NULL|
+---------+----------+------+----------+----+



In [12]:
#
## Replacing null values with some value
#
leftJoinDf.fillna(value="").show()

#
#leftJoinDf.fillna(value=0,subset=["customerId"]).show() # only for customerId column

#
# leftJoinDf.fillna(10,["customerId"]) \
    #.fillna("---",["name"]).show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  NULL|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|    |
|        5|      NULL|   150|      NULL|    |
|        6|       106|  NULL|       106|    |
+---------+----------+------+----------+----+



In [14]:
#
## Replacing null values with some value using na
#
leftJoinDf.na.fill(value=0).show()
leftJoinDf.na.fill(value=0,subset=["amount"]).show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|     0|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|         0|NULL|
|        5|         0|   150|         0|NULL|
|        6|       106|     0|       106|NULL|
+---------+----------+------+----------+----+

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|     0|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|      NULL|   150|      NULL|NULL|
|        6|       106|     0|       106|NULL|
+---------+----------+------+----------+----+



In [18]:
# Drop rows with nulls in any column
df_dropped_any = leftJoinDf.dropna(subset=["name"])

# Drop rows only if ALL columns are null
df_dropped_all = leftJoinDf.dropna(how="all")

# Drop rows with nulls only in the "Name" or "Score" columns
df_dropped_subset = leftJoinDf.dropna(subset=["customerId", "name"])


AnalysisException: [AMBIGUOUS_REFERENCE] Reference `customerId` is ambiguous, could be: [`customerId`, `customerId`].

In [5]:
from pyspark.sql.functions import col

# Keep rows where the "Name" column is not null
df_filtered_col = leftJoinDf.filter(col("name").isNotNull())
df_filtered_col.show()

# Alternatively, using dot notation (if column name has no spaces)
df_filtered_dot = leftJoinDf.filter(leftJoinDf.name.isNotNull())
df_filtered_dot.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+



In [7]:
# Filter using SQL expression
df_filtered_sql = leftJoinDf.filter("name IS NOT NULL")
df_filtered_sql.show()

# To also filter out empty strings, you can combine conditions customerId IS NOT NULL AND name != ''
df_filtered_sql_clean = leftJoinDf.filter("name != ''")
df_filtered_sql_clean.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|  2500|       101| Jon|
|        2|       102|  1110|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+

